# Fase 7 — Iteración: la feature de grupo y el asalto al 0.78+

El diagnóstico de la Fase 6 nos dejó tres líneas: (1) probar en el LB los modelos estables que nunca
subimos, (2) crear features de **supervivencia por grupo**, (3) gastar las submissions con hipótesis
claras. Esta fase ejecuta las tres — y de paso enseña la variante más sutil de *data leakage* de
todo el proyecto.

## La idea: las familias vivían o morían JUNTAS

Piensa en cómo ocurrió el naufragio: los grupos compartían camarote, se buscaban por el barco, subían
al mismo bote o se quedaban juntos esperando. El destino de tus compañeros de viaje es información
poderosa sobre el tuyo. Ya lo vimos en germen: los 11 Sage murieron los 11.

Y aquí está la jugada: **de los compañeros de grupo que cayeron en el train, sabemos qué les pasó.**
Para un pasajero del test cuya hermana está en el train y sobrevivió, ese dato es oro — y es
perfectamente legal usarlo: al predecir, los targets del train son conocidos.

Definimos `FamilySurvival` con tres valores:

| Valor | Significado |
|---|---|
| **1.0** | algún compañero de grupo conocido sobrevivió |
| **0.0** | todos sus compañeros conocidos murieron |
| **0.5** | sin información (viaja solo, o su grupo entero está en el test) |

Los grupos se detectan en dos pasadas: por **(apellido, fare)** — familias aunque lleven billetes
distintos — y por **ticket** — grupos no familiares (amigos, criados). Implementado en
[`src/grupos.py`](../src/grupos.py), con la regla crítica de **excluirse siempre a uno mismo** del
cálculo: si tu propia supervivencia entrara en tu feature, le estaríamos pasando la respuesta al
modelo dentro del enunciado.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import sys
sys.path.append("../src")
import validacion, grupos

train_l = pd.read_csv("../data/processed/train_limpio.csv")
test_l  = pd.read_csv("../data/processed/test_limpio.csv")
train_f = pd.read_csv("../data/processed/train_features.csv")
test_f  = pd.read_csv("../data/processed/test_features.csv")

X = train_f.drop(columns=["PassengerId", "Survived"])
y = train_f["Survived"]

# La feature sobre el train (cada pasajero excluido de su propio calculo)
fs_train = grupos.family_survival(train_l, train_l)
tabla = (train_f.assign(FamilySurvival=fs_train.loc[train_f.PassengerId].values)
         .groupby("FamilySurvival")["Survived"].agg(["count", "mean"]).round(3))
tabla.columns = ["pasajeros", "tasa_supervivencia"]
print(tabla)

fs_test = grupos.family_survival(train_l, test_l)
print("\nEn el test:", fs_test.value_counts().sort_index().to_dict(),
      "-> tenemos informacion de grupo para", (fs_test != 0.5).sum(), "de 418 pasajeros")

                pasajeros  tasa_supervivencia
FamilySurvival                               
0.0                   142               0.211
0.5                   609               0.353
1.0                   140               0.693



En el test: {0.0: 60, 0.5: 280, 1.0: 78} -> tenemos informacion de grupo para 138 de 418 pasajeros


**La señal es enorme:** si todos tus compañeros conocidos murieron → sobrevives el **21%** de las
veces; si alguno vivió → el **69%**. Y aplica a 138 pasajeros del test (un tercio). Compárala con
cualquier feature de la Fase 3: solo el sexo discrimina más.

## El leakage sutil: esta feature usa el TARGET

Aquí viene la parte delicada, y es la lección central de la fase. `FamilySurvival` se construye con
la columna `Survived` **de otros pasajeros**. Eso crea una trampa en la validación cruzada:

- **Lo correcto en producción/test:** usar los targets de TODO el train está bien — son conocidos al
  predecir.
- **La trampa en CV:** si calculamos la feature UNA vez con los 891 targets y luego hacemos CV,
  los pasajeros del fold de validación se delatan **entre sí** (tu hermana está en tu mismo fold de
  validación; su target alimentó tu feature → tu "examen" contiene un soplo). La CV saldría inflada.
- **Lo correcto en CV:** recalcular la feature **dentro de cada fold**, usando como "conocidos" solo
  los pasajeros del fold de entrenamiento — imitando exactamente la situación real del test.

`cross_val_score` no puede hacer esto por nosotros (no sabe que la feature depende del target), así
que escribimos el bucle de CV **a mano** — que además nos enseña qué hace `cross_val_score` por
dentro. Medimos también la versión mal hecha para cuantificar la trampa:

In [2]:
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

modelos = {
    "Logistica + FS": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "SVM + FS":       make_pipeline(StandardScaler(), SVC()),
    "KNN(9) + FS":    make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=9)),
    "RF afinado + FS": RandomForestClassifier(n_estimators=300, max_depth=8, max_features=0.5,
                                              min_samples_leaf=4, random_state=42),
    "GB afinado + FS": GradientBoostingClassifier(learning_rate=0.03, max_depth=4, n_estimators=100,
                                                  subsample=1.0, random_state=42),
}

# --- CV honesta: la feature se recalcula DENTRO de cada fold ---
scores = {n: [] for n in modelos}
for idx_tr, idx_va in validacion.CV.split(X, y):
    conocidos = train_l.iloc[idx_tr]                    # SOLO el fold de entrenamiento
    fs = grupos.family_survival(conocidos, train_l)     # feature para los 891 con esa info parcial
    X2 = X.assign(FamilySurvival=fs.loc[train_f.PassengerId].values)
    for nombre, modelo in modelos.items():
        m = clone(modelo).fit(X2.iloc[idx_tr], y.iloc[idx_tr])
        scores[nombre].append(m.score(X2.iloc[idx_va], y.iloc[idx_va]))

print("CV HONESTA (fold-aware):")
for nombre, s in scores.items():
    s = np.array(s)
    print(f"  {nombre:18} {s.mean():.4f} +- {s.std():.4f}")

# --- La version tramposa, solo para medir la trampa ---
from sklearn.model_selection import cross_val_score
X_mal = X.assign(FamilySurvival=fs_train.loc[train_f.PassengerId].values)
s_mal = cross_val_score(clone(modelos["SVM + FS"]), X_mal, y, cv=validacion.CV, n_jobs=-1)
print(f"\nSVM con feature calculada una sola vez (MAL): {s_mal.mean():.4f}  "
      f"(vs honesta {np.mean(scores['SVM + FS']):.4f})")

CV HONESTA (fold-aware):
  Logistica + FS     0.8451 +- 0.0230
  SVM + FS           0.8473 +- 0.0154
  KNN(9) + FS        0.8451 +- 0.0092
  RF afinado + FS    0.8507 +- 0.0137
  GB afinado + FS    0.8563 +- 0.0159



SVM con feature calculada una sola vez (MAL): 0.8496  (vs honesta 0.8473)


Resultados de la CV honesta, comparados con los mismos modelos sin la feature (Fases 4-5):

| Modelo | CV sin FS | CV con FS | Δ |
|---|---|---|---|
| Logística | 0.8328 | **0.8451** | +1.2 |
| SVM | 0.8350 | **0.8473** | +1.2 |
| RF afinado | 0.8473 | **0.8507** | +0.3 |
| GB afinado | 0.8563 | **0.8563** | = (pero σ baja de .024 a .016) |

Todos suben o se estabilizan. Y la trampa medida: la versión mal hecha inflaba el SVM solo ~0.2
puntos aquí (0.8496 vs 0.8473) — poca cosa *esta vez*, porque pocos familiares comparten fold de
validación; pero es pura suerte estructural. La disciplina de calcular features-con-target dentro
del fold es innegociable: cuando la trampa es grande, es invisible sin esta comparación.

## Las 5 submissions del día (cada una con su hipótesis)

| Versión | Qué prueba |
|---|---|
| v05 Logística sin FS | ¿Un modelo simple y estable aguanta mejor el salto CV→LB? |
| v06 SVM + FS | ¿La feature transfiere en un modelo de margen? |
| v07 RF + FS | ¿Y en bagging? |
| v08 KNN + FS | El clásico de esta feature en la comunidad (k vecinos con distancias) |
| v09 GB + FS | ¿Redime al caído de la Fase 6? |

Generación: mismo patrón de la Fase 6 (reentrenar con todo, validar con asserts, subir con el CLI).
Para el test, `FamilySurvival` se calcula con **todo el train** como conocidos — ahí es legítimo.

In [3]:
candidatos = {
    "v05_logistica": (make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), False),
    "v06_svm_fs":    (make_pipeline(StandardScaler(), SVC()), True),
    "v07_rf_fs":     (RandomForestClassifier(n_estimators=300, max_depth=8, max_features=0.5,
                                             min_samples_leaf=4, random_state=42), True),
    "v08_knn_fs":    (make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=9)), True),
    "v09_gb_fs":     (GradientBoostingClassifier(learning_rate=0.03, max_depth=4, n_estimators=100,
                                                 subsample=1.0, random_state=42), True),
}

X_fs      = X.assign(FamilySurvival=fs_train.loc[train_f.PassengerId].values)
X_test    = test_f[X.columns]
X_test_fs = X_test.assign(FamilySurvival=fs_test.loc[test_f.PassengerId].values)

for nombre, (modelo, con_fs) in candidatos.items():
    Xtr, Xte = (X_fs, X_test_fs) if con_fs else (X, X_test)
    p = modelo.fit(Xtr, y).predict(Xte)
    sub = pd.DataFrame({"PassengerId": test_f["PassengerId"], "Survived": p})
    assert sub.shape == (418, 2) and sub["Survived"].isin([0, 1]).all() and sub.isna().sum().sum() == 0
    sub.to_csv(f"../submissions/submission_{nombre}.csv", index=False)
    print(f"{nombre}: OK (tasa predicha {p.mean():.3f})")

v05_logistica: OK (tasa predicha 0.409)
v06_svm_fs: OK (tasa predicha 0.364)


v07_rf_fs: OK (tasa predicha 0.364)
v08_knn_fs: OK (tasa predicha 0.402)


v09_gb_fs: OK (tasa predicha 0.388)


Subidas el 2026-07-08 con `kaggle competitions submit` (5 de las 8 que quedaban). Y el veredicto:

## El veredicto del leaderboard

| Versión | CV honesta | **LB público** | Salto vs 0.76794 |
|---|---|---|---|
| **v06 SVM + FamilySurvival** | 0.8473 ± 0.015 | **0.79425** 🏆 | **+2.6 pts** |
| v07 RF afinado + FamilySurvival | 0.8507 ± 0.014 | 0.78468 | +1.7 pts |
| v08 KNN(9) + FamilySurvival | 0.8451 ± 0.009 | 0.77990 | +1.2 pts |
| v09 GB afinado + FamilySurvival | 0.8563 ± 0.016 | 0.77511 | +0.7 pts |
| v05 Logística (sin FS) | 0.8328 ± 0.011 | 0.76794 | +0.0 |

Lecturas, de mayor a menor calado:

1. **La feature transfirió al mundo real en las CUATRO familias de modelos** (+0.7 a +2.6 puntos).
   Esto es lo que distingue una mejora genuina de un espejismo de CV: robustez entre modelos y
   entre conjuntos. Las mejores features codifican *mecanismos* del mundo real (los grupos
   compartían destino), no correlaciones estadísticas frágiles.
2. **Meta del plan alcanzada:** 0.79425, dentro del rango objetivo 0.78-0.80. De la regla del género
   (0.7655) al 0.794: +2.9 puntos, de los cuales ~2.6 los puso la feature de grupo — y casi nada el
   tuning de modelos. La moraleja número uno del proyecto, confirmada empíricamente:
   **en tabular, las features mandan.**
3. **El orden CV ≠ orden LB, otra vez:** ganó el SVM (3º por CV), y el GB volvió a quedar último de
   los cuatro pese a su CV de 0.8563. Segunda vez que el GB promete y decepciona — patrón, no azar:
   su capacidad extra memoriza matices del train que el test no comparte.
4. **v05 (logística sola): 0.76794, EXACTAMENTE igual que v02 (RF sin FS).** Difieren en 34
   pasajeros y empatan al punto: fallan lo mismo en total, en gente distinta. Sin la feature de
   grupo, ambos habían extraído ya toda la señal disponible — el techo de las 16 features era ese.

## Registro final

In [4]:
registro = pd.read_csv("../submissions/registro_experimentos.csv")
nuevos = pd.DataFrame([
    {"version": "v05-logistica",  "fecha": "2026-07-08", "features": "16 (Fase 3)",      "modelo": "StandardScaler+LogisticRegression", "cv_media": 0.8328, "cv_std": 0.0110, "lb_publico": 0.76794},
    {"version": "v06-svm-fs",     "fecha": "2026-07-08", "features": "16 + FamilySurvival", "modelo": "StandardScaler+SVC",             "cv_media": 0.8473, "cv_std": 0.0154, "lb_publico": 0.79425},
    {"version": "v07-rf-fs",      "fecha": "2026-07-08", "features": "16 + FamilySurvival", "modelo": "RF(depth=8, leaf=4, feat=0.5)",  "cv_media": 0.8507, "cv_std": 0.0137, "lb_publico": 0.78468},
    {"version": "v08-knn-fs",     "fecha": "2026-07-08", "features": "16 + FamilySurvival", "modelo": "KNN(k=9) escalado",              "cv_media": 0.8451, "cv_std": 0.0092, "lb_publico": 0.77990},
    {"version": "v09-gb-fs",      "fecha": "2026-07-08", "features": "16 + FamilySurvival", "modelo": "GB(lr=0.03, depth=4, n=100)",    "cv_media": 0.8563, "cv_std": 0.0159, "lb_publico": 0.77511},
])
registro = registro[~registro["version"].isin(nuevos["version"])]
registro = pd.concat([registro, nuevos], ignore_index=True)
registro.to_csv("../submissions/registro_experimentos.csv", index=False)
registro.sort_values("lb_publico", ascending=False, na_position="last")

,version,fecha,features,modelo,cv_media,cv_std,lb_publico
7,v06-svm-fs,2026-07-08,16 + FamilySurvival,StandardScaler+SVC,0.8473,0.0154,0.79425
8,v07-rf-fs,2026-07-08,16 + FamilySurvival,"RF(depth=8, leaf=4, feat=0.5)",0.8507,0.0137,0.78468
9,v08-knn-fs,2026-07-08,16 + FamilySurvival,KNN(k=9) escalado,0.8451,0.0092,0.77990
10,v09-gb-fs,2026-07-08,16 + FamilySurvival,"GB(lr=0.03, depth=4, n=100)",0.8563,0.0159,0.77511
6,v05-logistica,2026-07-08,16 (Fase 3),StandardScaler+LogisticRegression,0.8328,0.0110,0.76794
3,v02-rf-afinado,2026-07-08,16 (Fase 3),"RF(depth=8, leaf=4, feat=0.5)",0.8473,0.0092,0.76794
1,base-genero,2026-07-08,IsFemale,ReglaGenero,0.7868,0.0188,0.76550
4,v03-gb-afinado,2026-07-08,16 (Fase 3),"GB(lr=0.03, depth=4, n=100)",0.8563,0.0245,0.74880
0,base-mayoria,2026-07-08,-,DummyClassifier(most_frequent),0.6162,0.0023,NaN
2,v01-logistica,2026-07-08,16 (Fase 3),StandardScaler+LogisticRegression,0.8328,0.0110,NaN


## El viaje completo del proyecto

| Hito | Herramienta | LB |
|---|---|---|
| Suelo absoluto ("nadie vive") | — | ~0.62 (estimado) |
| Regla del género | 1 columna | 0.7655 |
| Mejor modelo, 16 features (Fases 2-5) | RF afinado | 0.76794 |
| **+ FamilySurvival (Fase 7)** | **SVM** | **0.79425** |

Y las lecciones que valen para cualquier proyecto futuro, por orden de importancia:

1. **Las features ganan a los modelos.** +2.6 puntos por una feature con mecanismo real; +0.002 por
   todo el torneo y tuning de la Fase 5.
2. **El leakage tiene mil caras** — medianas del test (Fase 2), el scaler en la CV (Fase 4), tu
   propio target en una feature de grupo (hoy). El antídoto es siempre el mismo: pregúntate *"¿esta
   información existiría en el momento real de predecir?"* y estructura el código (fit/transform,
   pipelines, CV fold-aware) para que la respuesta sea sí.
3. **Respeta la varianza.** El GB ganó dos veces en CV media y perdió dos veces en el LB. Media sin
   desviación es media verdad.
4. **Protocolo antes que resultados**: juez fijo (CV sembrada), registro de experimentos, hipótesis
   por submission. Es lo que permite *aprender* de cada resultado en vez de deambular.

### ¿Y si quisiéramos más?

El rango honesto por encima de 0.794 existe pero se estrecha: reglas puras de grupo mujer-niño
(*woman-child groups*, ~0.80-0.82), interacciones más finas, stacking disciplinado. A partir de ~0.83
ya sabes lo que hay (Fase 6 de este notebook... y la charla sobre el 100%). Para un primer proyecto
completo de ML, **0.79425 con cada decisión justificada y reproducible** es exactamente lo que había
que conseguir. 🚢